# AV2 – Análise Estatística e Testes de Hipótese
## House Prices – Ames, Iowa

Este notebook aprofunda a análise exploratória realizada na AV1, formalizando os testes estatísticos para validar (ou refutar) as hipóteses levantadas.

---
## 0. Importações e Carregamento dos Dados

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.weightstats import zconfint
import warnings
warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='coolwarm')
plt.rcParams['figure.dpi'] = 100

# Carregamento
df = pd.read_csv('../data/raw/train.csv')

# Pré-processamento básico (igual à AV1)
df.fillna(df.mean(numeric_only=True), inplace=True)
df.drop_duplicates(inplace=True)
df.columns = df.columns.str.strip()

# Coluna auxiliar de reforma
df['Foi_Reformada'] = np.where(df['YearRemodAdd'] > df['YearBuilt'], 'Sim', 'Não')

print(f'Dataset carregado: {df.shape[0]} imóveis | {df.shape[1]} variáveis')
df[['SalePrice', 'GrLivArea', 'YearBuilt', 'Foi_Reformada', 'Neighborhood']].head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/train.csv'

---
## 1. Teste de Normalidade do Preço de Venda (SalePrice)

**Por que isso importa?**  
Muitos testes paramétricos (como o Teste t e a ANOVA) assumem que os dados seguem uma distribuição normal. Antes de aplicá-los, precisamos verificar esse pressuposto.

**Hipóteses:**
- **H₀:** SalePrice segue uma distribuição normal  
- **H₁:** SalePrice NÃO segue uma distribuição normal

In [8]:
# --- Teste de Shapiro-Wilk (indicado para n < 5000) ---
# Usamos uma amostra de 2000 para o Shapiro-Wilk (limite recomendado)
amostra = df['SalePrice'].sample(2000, random_state=42)
stat_sw, p_sw = stats.shapiro(amostra)

# --- Teste de Kolmogorov-Smirnov ---
stat_ks, p_ks = stats.kstest(df['SalePrice'], 'norm',
                              args=(df['SalePrice'].mean(), df['SalePrice'].std()))

print('=== TESTE DE NORMALIDADE – SalePrice ===')
print(f'\nShapiro-Wilk   | Estatística: {stat_sw:.4f} | p-valor: {p_sw:.6f}')
print(f'Kolmogorov-Smirnov | Estatística: {stat_ks:.4f} | p-valor: {p_ks:.6f}')

if p_sw < 0.05:
    print('\n➜ Resultado: p < 0.05 → Rejeitamos H₀. SalePrice NÃO é normal.')
else:
    print('\n➜ Resultado: p ≥ 0.05 → Não rejeitamos H₀. SalePrice pode ser normal.')

# --- Visualização ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma + curva normal
sns.histplot(df['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição do SalePrice (original)', fontsize=13)
axes[0].set_xlabel('Preço de Venda ($)')

# Q-Q Plot
stats.probplot(df['SalePrice'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot – SalePrice (original)', fontsize=13)

plt.tight_layout()
plt.show()

ValueError: Cannot take a larger sample than population when 'replace=False'

### 1.1 Transformação Logarítmica e Novo Teste de Normalidade

Como o SalePrice apresenta assimetria positiva (cauda à direita), aplicamos a transformação log para aproximá-lo da normalidade.

In [ ]:
df['LogSalePrice'] = np.log(df['SalePrice'])

# Novos testes com log
amostra_log = df['LogSalePrice'].sample(2000, random_state=42)
stat_sw_log, p_sw_log = stats.shapiro(amostra_log)
stat_ks_log, p_ks_log = stats.kstest(df['LogSalePrice'], 'norm',
                                      args=(df['LogSalePrice'].mean(), df['LogSalePrice'].std()))

print('=== TESTE DE NORMALIDADE – log(SalePrice) ===')
print(f'\nShapiro-Wilk   | Estatística: {stat_sw_log:.4f} | p-valor: {p_sw_log:.6f}')
print(f'Kolmogorov-Smirnov | Estatística: {stat_ks_log:.4f} | p-valor: {p_ks_log:.6f}')

# Skewness antes/depois
print(f'\nAssimetria (Skewness) original: {df["SalePrice"].skew():.4f}')
print(f'Assimetria (Skewness) log:      {df["LogSalePrice"].skew():.4f}')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['LogSalePrice'], kde=True, ax=axes[0], color='seagreen')
axes[0].set_title('Distribuição do log(SalePrice)', fontsize=13)
axes[0].set_xlabel('log(Preço de Venda)')

stats.probplot(df['LogSalePrice'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot – log(SalePrice)', fontsize=13)

plt.tight_layout()
plt.show()

---
## 2. Casas Reformadas Valem Mais? – Teste t de Student

Na AV1 comparamos as medianas visualmente. Agora formalizamos com um **Teste t de Welch** (não assume variâncias iguais entre os grupos).

**Hipóteses:**
- **H₀:** A média de preço de casas reformadas = média de casas não reformadas  
- **H₁:** A média de preço de casas reformadas ≠ média de casas não reformadas  
- **Nível de significância:** α = 0.05

In [ ]:
# --- Verificação de Homogeneidade de Variâncias (Levene) ---
grupo_sim = df[df['Foi_Reformada'] == 'Sim']['SalePrice']
grupo_nao = df[df['Foi_Reformada'] == 'Não']['SalePrice']

stat_levene, p_levene = stats.levene(grupo_sim, grupo_nao)
print('--- Teste de Levene (Homogeneidade de Variâncias) ---')
print(f'Estatística: {stat_levene:.4f} | p-valor: {p_levene:.6f}')
if p_levene < 0.05:
    print('➜ Variâncias DIFERENTES → usamos Teste t de Welch (equal_var=False)\n')
else:
    print('➜ Variâncias IGUAIS → podemos usar Teste t padrão\n')

# --- Teste t de Welch ---
t_stat, p_valor_t = stats.ttest_ind(grupo_sim, grupo_nao, equal_var=False)

print('=== TESTE T DE WELCH – Reformada vs. Não Reformada ===')
print(f'Estatística t: {t_stat:.4f}')
print(f'p-valor:       {p_valor_t:.6f}')
print(f'\nMédia – Reformada:     ${grupo_sim.mean():,.2f}')
print(f'Média – Não Reformada: ${grupo_nao.mean():,.2f}')
print(f'Diferença de médias:   ${grupo_sim.mean() - grupo_nao.mean():,.2f}')

if p_valor_t < 0.05:
    print('\n➜ Resultado: p < 0.05 → Rejeitamos H₀.')
    print('   Casas reformadas têm preço médio ESTATISTICAMENTE DIFERENTE das não reformadas.')
else:
    print('\n➜ Resultado: p ≥ 0.05 → Não rejeitamos H₀.')
    print('   Não há evidência estatística de diferença entre os grupos.')

# --- Intervalo de Confiança (95%) para a diferença das médias ---
n1, n2 = len(grupo_sim), len(grupo_nao)
mean_diff = grupo_sim.mean() - grupo_nao.mean()
se_diff = np.sqrt(grupo_sim.var()/n1 + grupo_nao.var()/n2)
t_crit = stats.t.ppf(0.975, df=min(n1, n2) - 1)
ic_low = mean_diff - t_crit * se_diff
ic_high = mean_diff + t_crit * se_diff
print(f'\nIC 95% para (Reformada − Não Reformada): [${ic_low:,.2f}, ${ic_high:,.2f}]')

# --- Visualização ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Foi_Reformada', y='SalePrice',
            palette=['lightcoral', 'lightblue'], ax=axes[0])
axes[0].set_title('Distribuição de Preços: Reformada vs. Não Reformada', fontsize=12)
axes[0].set_xlabel('Foi Reformada?')
axes[0].set_ylabel('Preço de Venda ($)')

sns.violinplot(data=df, x='Foi_Reformada', y='SalePrice',
               palette=['lightcoral', 'lightblue'], ax=axes[1], inner='quartile')
axes[1].set_title('Violin Plot: Distribuição por Grupo', fontsize=12)
axes[1].set_xlabel('Foi Reformada?')
axes[1].set_ylabel('Preço de Venda ($)')

plt.tight_layout()
plt.show()

---
## 3. Bairros Têm Preços Diferentes? – ANOVA + Post-hoc Tukey HSD

Na AV1 aplicamos a ANOVA e concluímos que há diferença. Agora identificamos **quais pares de bairros** diferem entre si com o **Teste de Tukey HSD**.

**Hipóteses:**
- **H₀:** As médias de SalePrice são iguais em todos os bairros  
- **H₁:** Pelo menos um bairro tem média diferente dos demais

In [ ]:
# --- ANOVA ---
grupos_bairros = [df[df['Neighborhood'] == b]['SalePrice']
                  for b in df['Neighborhood'].unique()]
f_stat, p_anova = stats.f_oneway(*grupos_bairros)

print('=== ANOVA – Preço por Bairro ===')
print(f'Estatística F: {f_stat:.4f}')
print(f'p-valor:       {p_anova:.2e}')

if p_anova < 0.05:
    print('➜ Rejeitamos H₀ → Há diferença estatística entre os bairros.')
    print('  Aplicando Post-hoc Tukey HSD para identificar os pares...\n')

# --- Tukey HSD ---
tukey = pairwise_tukeyhsd(endog=df['SalePrice'],
                           groups=df['Neighborhood'],
                           alpha=0.05)

# Resumo: pares com diferença significativa
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                         columns=tukey._results_table.data[0])
tukey_df['reject'] = tukey_df['reject'].astype(bool)
significativos = tukey_df[tukey_df['reject'] == True]

print(f'Pares de bairros com diferença SIGNIFICATIVA: {len(significativos)} de {len(tukey_df)}')
print('\nTop 10 maiores diferenças de preço médio entre bairros:')
significativos['meandiff'] = significativos['meandiff'].astype(float)
print(significativos.nlargest(10, 'meandiff')[['group1', 'group2', 'meandiff', 'p-adj']].to_string(index=False))

# --- Visualização: Top 5 e Bottom 5 bairros ---
media_bairro = df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)
top5 = media_bairro.head(5).index
bot5 = media_bairro.tail(5).index
bairros_selecionados = list(top5) + list(bot5)

df_sel = df[df['Neighborhood'].isin(bairros_selecionados)]

plt.figure(figsize=(14, 6))
ordem = media_bairro[bairros_selecionados].sort_values(ascending=False).index
sns.boxplot(data=df_sel, x='Neighborhood', y='SalePrice', order=ordem,
            palette='RdYlGn')
plt.title('Top 5 Bairros Mais Caros vs. 5 Mais Baratos', fontsize=14)
plt.xlabel('Bairro')
plt.ylabel('Preço de Venda ($)')
plt.xticks(rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()

---
## 4. Área Construída Tem Relação Linear com Preço? – Correlação de Pearson

Na AV1 fizemos o scatter plot. Agora testamos se a correlação é estatisticamente significativa.

**Hipóteses:**
- **H₀:** A correlação entre GrLivArea e SalePrice é zero (ρ = 0)  
- **H₁:** A correlação entre GrLivArea e SalePrice é diferente de zero (ρ ≠ 0)

In [ ]:
# --- Correlação de Pearson ---
r_pearson, p_pearson = stats.pearsonr(df['GrLivArea'], df['SalePrice'])

# --- Correlação de Spearman (robusta a não-linearidade) ---
r_spearman, p_spearman = stats.spearmanr(df['GrLivArea'], df['SalePrice'])

print('=== CORRELAÇÃO: GrLivArea vs. SalePrice ===')
print(f'\nPearson  | r = {r_pearson:.4f} | p-valor: {p_pearson:.2e}')
print(f'Spearman | ρ = {r_spearman:.4f} | p-valor: {p_spearman:.2e}')

if p_pearson < 0.05:
    print(f'\n➜ Rejeitamos H₀. Há correlação SIGNIFICATIVA (r = {r_pearson:.4f}).')
    if abs(r_pearson) >= 0.7:
        print('   Força da correlação: FORTE')
    elif abs(r_pearson) >= 0.4:
        print('   Força da correlação: MODERADA')
    else:
        print('   Força da correlação: FRACA')

# --- IC 95% para r de Pearson (transformação de Fisher) ---
n = len(df)
z = np.arctanh(r_pearson)
se = 1 / np.sqrt(n - 3)
z_crit = stats.norm.ppf(0.975)
ic_r_low = np.tanh(z - z_crit * se)
ic_r_high = np.tanh(z + z_crit * se)
print(f'\nIC 95% para r de Pearson: [{ic_r_low:.4f}, {ic_r_high:.4f}]')

# --- Visualização ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(data=df, x='GrLivArea', y='SalePrice', ax=axes[0],
            scatter_kws={'alpha': 0.4, 'color': 'steelblue'},
            line_kws={'color': 'red'})
axes[0].set_title(f'GrLivArea vs. SalePrice\n(Pearson r = {r_pearson:.4f})', fontsize=12)
axes[0].set_xlabel('Área do Living Room (sq ft)')
axes[0].set_ylabel('Preço de Venda ($)')

sns.regplot(data=df, x='GrLivArea', y='LogSalePrice', ax=axes[1],
            scatter_kws={'alpha': 0.4, 'color': 'seagreen'},
            line_kws={'color': 'red'})
r_log, _ = stats.pearsonr(df['GrLivArea'], df['LogSalePrice'])
axes[1].set_title(f'GrLivArea vs. log(SalePrice)\n(Pearson r = {r_log:.4f})', fontsize=12)
axes[1].set_xlabel('Área do Living Room (sq ft)')
axes[1].set_ylabel('log(Preço de Venda)')

plt.tight_layout()
plt.show()

---
## 5. Ano de Construção Influencia o Preço? – Correlação de Spearman

Como a relação entre ano e preço pode ser não-linear, usamos o **Coeficiente de Spearman**, que mede associação monotônica sem exigir linearidade.

**Hipóteses:**
- **H₀:** Não há associação monotônica entre YearBuilt e SalePrice (ρ = 0)  
- **H₁:** Existe associação monotônica entre YearBuilt e SalePrice (ρ ≠ 0)

In [ ]:
# --- Spearman: YearBuilt vs SalePrice ---
r_sp_year, p_sp_year = stats.spearmanr(df['YearBuilt'], df['SalePrice'])
r_pe_year, p_pe_year = stats.pearsonr(df['YearBuilt'], df['SalePrice'])

print('=== CORRELAÇÃO: YearBuilt vs. SalePrice ===')
print(f'\nPearson  | r = {r_pe_year:.4f} | p-valor: {p_pe_year:.2e}')
print(f'Spearman | ρ = {r_sp_year:.4f} | p-valor: {p_sp_year:.2e}')

if p_sp_year < 0.05:
    print(f'\n➜ Rejeitamos H₀. O ano de construção TEM associação significativa com o preço.')
    if r_sp_year > 0:
        print('   Direção: POSITIVA → casas mais novas tendem a ser mais caras.')
    else:
        print('   Direção: NEGATIVA → casas mais antigas tendem a ser mais caras.')

# --- Visualização ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

preco_por_ano = df.groupby('YearBuilt')['SalePrice'].mean().reset_index()
sns.lineplot(data=preco_por_ano, x='YearBuilt', y='SalePrice',
             color='darkblue', linewidth=2, ax=axes[0])
axes[0].set_title(f'Preço Médio por Ano de Construção\n(Spearman ρ = {r_sp_year:.4f})', fontsize=12)
axes[0].set_xlabel('Ano de Construção')
axes[0].set_ylabel('Preço Médio ($)')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

sns.regplot(data=df, x='YearBuilt', y='SalePrice', ax=axes[1],
            scatter_kws={'alpha': 0.3, 'color': 'slateblue'},
            line_kws={'color': 'red'})
axes[1].set_title('Dispersão: YearBuilt vs. SalePrice', fontsize=12)
axes[1].set_xlabel('Ano de Construção')
axes[1].set_ylabel('Preço de Venda ($)')

plt.tight_layout()
plt.show()

---
## 6. Intervalo de Confiança (95%) para a Média do SalePrice

Estimamos, com 95% de confiança, o intervalo em que está a média real do preço de venda dos imóveis na população.

In [ ]:
from scipy.stats import t as t_dist

def intervalo_confianca(serie, confianca=0.95):
    n = len(serie)
    media = serie.mean()
    se = stats.sem(serie)
    margem = se * t_dist.ppf((1 + confianca) / 2, df=n - 1)
    return media, media - margem, media + margem

# IC geral
media_geral, ic_low_g, ic_high_g = intervalo_confianca(df['SalePrice'])
print('=== INTERVALO DE CONFIANÇA 95% – SalePrice ===')
print(f'Média amostral: ${media_geral:,.2f}')
print(f'IC 95%: [${ic_low_g:,.2f}, ${ic_high_g:,.2f}]')

# IC por grupo de reforma
print('\n--- IC 95% por Grupo de Reforma ---')
for grupo in ['Sim', 'Não']:
    dados = df[df['Foi_Reformada'] == grupo]['SalePrice']
    m, low, high = intervalo_confianca(dados)
    print(f'Reformada = {grupo}: média ${m:,.2f} | IC: [${low:,.2f}, ${high:,.2f}] | n={len(dados)}')

# IC Top 5 bairros
print('\n--- IC 95% – Top 5 Bairros Mais Caros ---')
for bairro in top5:
    dados = df[df['Neighborhood'] == bairro]['SalePrice']
    m, low, high = intervalo_confianca(dados)
    print(f'{bairro:15s}: média ${m:,.2f} | IC: [${low:,.2f}, ${high:,.2f}] | n={len(dados)}')

# Visualização: ICs dos grupos de reforma
grupos_ic = []
for grupo in df['Foi_Reformada'].unique():
    dados = df[df['Foi_Reformada'] == grupo]['SalePrice']
    m, low, high = intervalo_confianca(dados)
    grupos_ic.append({'Grupo': f'Reformada = {grupo}', 'Média': m, 'IC_Low': low, 'IC_High': high})

ic_df = pd.DataFrame(grupos_ic)

plt.figure(figsize=(8, 4))
plt.errorbar(ic_df['Grupo'], ic_df['Média'],
             yerr=[ic_df['Média'] - ic_df['IC_Low'], ic_df['IC_High'] - ic_df['Média']],
             fmt='o', capsize=10, color='steelblue', ecolor='gray', markersize=10)
plt.title('Intervalo de Confiança 95% – Preço por Grupo de Reforma', fontsize=13)
plt.ylabel('Preço de Venda ($)')
plt.xlabel('')
plt.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

---
## 7. Resumo Geral dos Testes Estatísticos

In [ ]:
resumo = pd.DataFrame([
    {
        'Hipótese': 'SalePrice tem distribuição normal?',
        'Teste': 'Shapiro-Wilk',
        'p-valor': f'{p_sw:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀',
        'Conclusão': 'SalePrice NÃO é normal (assimetria positiva)'
    },
    {
        'Hipótese': 'log(SalePrice) tem distribuição normal?',
        'Teste': 'Shapiro-Wilk',
        'p-valor': f'{p_sw_log:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀' if p_sw_log < 0.05 else 'Não rejeita H₀',
        'Conclusão': 'Transformação log reduz assimetria significativamente'
    },
    {
        'Hipótese': 'Casas reformadas valem mais?',
        'Teste': 'Teste t de Welch',
        'p-valor': f'{p_valor_t:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀' if p_valor_t < 0.05 else 'Não rejeita H₀',
        'Conclusão': 'Sim, diferença estatisticamente significativa'
    },
    {
        'Hipótese': 'Bairros têm preços diferentes?',
        'Teste': 'ANOVA + Tukey HSD',
        'p-valor': f'{p_anova:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀',
        'Conclusão': f'Sim, {len(significativos)} pares de bairros diferem significativamente'
    },
    {
        'Hipótese': 'GrLivArea tem correlação com SalePrice?',
        'Teste': 'Pearson + Spearman',
        'p-valor': f'{p_pearson:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀',
        'Conclusão': f'Sim, correlação forte (r = {r_pearson:.4f})'
    },
    {
        'Hipótese': 'Ano de construção associado ao preço?',
        'Teste': 'Spearman',
        'p-valor': f'{p_sp_year:.2e}',
        'Decisão (α=0.05)': 'Rejeita H₀' if p_sp_year < 0.05 else 'Não rejeita H₀',
        'Conclusão': f'Sim, associação positiva (ρ = {r_sp_year:.4f})'
    },
])

pd.set_option('display.max_colwidth', 60)
display(resumo)